## 1. Console Debugging

The most immediate tool: `console.log` to inspect state while code runs.

From the game engine — `Coin.js` logs creation and collection events:

```js
// Coin.js — strategic logging in the constructor
constructor(data = null, gameEnv = null) {
    super(coinData, gameEnv);
    this.value = Number(data?.value ?? 1);
    console.log('Coin created:', this.spriteData.id, 'at position', this.position); // ← constructor log
}

// In update() — logs every collision event
handleCollisionEvent() {
    console.log('Shark collided with player!');  // ← method-level log
    this.velocity.x = 0;
    this.velocity.y = 0;
}
```

**Console methods:**
- `console.log(msg)` — general info
- `console.warn(msg)` — yellow warning
- `console.error(msg)` — red error  
- `console.table(array)` — display arrays/objects as a table
- `console.group('label')` / `console.groupEnd()` — collapsible groups

In [ ]:
%%js
//CODE_RUNNER: Console debugging practice. Track game state through a simulated update() loop.

class DebugPlayer {
    constructor(name) {
        this.name   = name;
        this.x      = 0;
        this.y      = 300;
        this.health = 100;
        this.score  = 0;
        console.log(`[INIT] Player '${name}' created at (${this.x}, ${this.y})`);
    }

    update(frame) {
        this.x += 5;
        if (frame === 3) {
            this.health -= 20;
            console.warn(`[FRAME ${frame}] Player hit! Health: ${this.health}`);
        }
        if (frame % 2 === 0) {
            this.score += 10;
        }
        console.log(`[FRAME ${frame}] pos=(${this.x}, ${this.y}) hp=${this.health} score=${this.score}`);
    }
}

const player = new DebugPlayer('Seonyoo');
for (let frame = 1; frame <= 6; frame++) {
    player.update(frame);
}

console.table({
    name:   player.name,
    finalX: player.x,
    health: player.health,
    score:  player.score
});


## 2. Hit Box Visualization

Collision bugs are hard to fix when you can't see the boxes. The game engine stores hitbox data, and you can draw them on the canvas to verify they line up:

```js
// Drawing a hitbox overlay in draw()
draw() {
    // Normal sprite drawing ...
    this.ctx.drawImage(this.spriteSheet, ...);

    // Toggle hitbox display for debugging
    if (this.showHitbox) {
        const hitW = this.width  * this.hitbox.widthPercentage;
        const hitH = this.height * this.hitbox.heightPercentage;
        const hitX = (this.width  - hitW) / 2;
        const hitY = (this.height - hitH) / 2;
        this.ctx.strokeStyle = 'rgba(255, 0, 0, 0.8)';  // red hitbox
        this.ctx.lineWidth = 2;
        this.ctx.strokeRect(hitX, hitY, hitW, hitH);
    }
}
```

From player config in `game-runner-example.ipynb`:
```js
hitbox: { widthPercentage: 0.45, heightPercentage: 0.2 }
```
Adjust these values until the visible red box matches the sprite.

In [ ]:
%%js
//CODE_RUNNER: Hitbox visualizer. Adjust widthPct and heightPct to align the red box with the blue sprite.

const canvas = document.createElement('canvas');
canvas.width  = 300;
canvas.height = 200;
canvas.style.border     = '2px solid #555';
canvas.style.background = '#1a1a2e';
document.body.appendChild(canvas);
const ctx = canvas.getContext('2d');

// Sprite dimensions (simulated character)
const spriteW = 60;
const spriteH = 80;
const spriteX = 120;
const spriteY = 60;

// --- ADJUST THESE to tune the hitbox ---
const widthPct  = 0.6;   // try 0.45 (game default for player)
const heightPct = 0.4;   // try 0.2

// Draw sprite (blue rectangle as stand-in)
ctx.fillStyle = '#4a9eff';
ctx.fillRect(spriteX, spriteY, spriteW, spriteH);

// Draw hitbox overlay
const hitW = spriteW * widthPct;
const hitH = spriteH * heightPct;
const hitX = spriteX + (spriteW - hitW) / 2;
const hitY = spriteY + (spriteH - hitH) / 2;

ctx.strokeStyle = 'rgba(255, 0, 0, 0.9)';
ctx.lineWidth   = 2;
ctx.strokeRect(hitX, hitY, hitW, hitH);

// Labels
ctx.fillStyle = 'white';
ctx.font      = '11px monospace';
ctx.fillText(`Hitbox: ${Math.round(hitW)}x${Math.round(hitH)}`, 10, 20);
ctx.fillText(`widthPct=${widthPct} heightPct=${heightPct}`, 10, 36);

console.log(`Hitbox size: ${Math.round(hitW)} x ${Math.round(hitH)} px`);


## 3. DevTools — Source, Network & Application

Open DevTools with **F12** (or Cmd+Option+I on Mac).

### Sources Tab — Breakpoints
1. Open **Sources** → find your JS file in the file tree  
2. Click a line number to set a **breakpoint** (blue marker appears)  
3. Trigger the code (move the player, collect a coin)  
4. Execution pauses — use **Step Over (F10)** / **Step Into (F11)** to walk through code  
5. Hover variables or use the **Scope** panel to inspect values  

### Network Tab — API Calls
1. Open **Network** and reload the page  
2. Look for `fetch` requests to `flask.opencodingsociety.com` or `spring.opencodingsociety.com`  
3. Click a request → check **Status** (200 = OK, 401 = not logged in, 403 = forbidden, 500 = server error)  
4. Check **Headers** for CORS errors: look for `Access-Control-Allow-Origin`  
5. Click **Response** to see the raw JSON the server returned  

### Application Tab — Cookies & localStorage
1. Open **Application** → **Cookies** → select your site  
2. Look for the JWT cookie (`jwt_java_spring` or `jwt_ms_flask`)  
3. If missing → user is not logged in  
4. Check **localStorage** for any client-side stored state (game progress, preferences)  

### Elements Tab — Inspect Canvas & DOM
1. Right-click any canvas → **Inspect**  
2. See the canvas id, width/height attributes  
3. Change CSS in the **Styles** panel live to test layout fixes

## Debugging Checklist

| Tool | Where | What to look for |
|---|---|---|
| Console | Console tab | Errors, logs from `update()` and `collisionHandler()` |
| Hitbox overlay | Canvas | Red box aligns with visible sprite |
| Breakpoints | Sources tab | Pause at collision, inspect `this.position`, `this.velocity` |
| Network tab | Network tab | 200 OK on fetch calls; CORS headers present |
| Application tab | Application tab | JWT cookie exists after login |
| Element inspector | Elements tab | Canvas size, game container styles |